GXM-01 — Machine Learning Model Development

Objective

Develop and evaluate a binary classification model using the Breast Cancer Wisconsin (Diagnostic) dataset.

This notebook covers:

Data preparation
Feature and target separtion
Stratified train-test splitting
Feature scaling 
Logistic Regression
Model evaluation

The dataset is used for educational and research purposes, not clinical diagnosis.

In [19]:
import pandas as pd 
import numpy as np

from sklearn.datasets import load_breast_cancer
from sklearn.model_selection import train_test_split

#Load dataset
dataset = load_breast_cancer()

#Convert features to DataFrame
df = pd.DataFrame(dataset.data,columns=dataset.feature_names)

#Add target
df["target"] = dataset.target

print("Dataset shape:" ,df.shape)


Dataset shape: (569, 31)


In [20]:
# Input features
X = df.drop(columns=["target"])

# Target variable
y = df["target"]

print ("Feature shape:", X.shape)
print ("Target shape:",y.shape)

Feature shape: (569, 30)
Target shape: (569,)


In [21]:
X_train,X_test,y_train,y_test = train_test_split(X,y,test_size=0.20,random_state=42,stratify=y)

print("Training samples:", X_train.shape[0])
print("Testing samples:",X_test.shape[0])

Training samples: 455
Testing samples: 114


In [22]:
print("Original distribution:")
print(y.value_counts(normalize=True))

print("\nTraining distribution:")
print(y_train.value_counts(normalize=True))

print("\nTesting distribution:")
print(y_test.value_counts(normalize=True))

Original distribution:
target
1    0.627417
0    0.372583
Name: proportion, dtype: float64

Training distribution:
target
1    0.626374
0    0.373626
Name: proportion, dtype: float64

Testing distribution:
target
1    0.631579
0    0.368421
Name: proportion, dtype: float64


Observation - Train-Test Split

the dataset was divided into 80% training data (455 samples) and 20 % testing samples (114 samples).

Stratified sampling was applied to maintain approcimately the same benign-to-malignant class distribution across both subsets

A fixed random state was used to ensure reproducibility. The test dataset will remain separate during model development and will be used for final model evaluation.


Feature Scaling 

StandardScaler is used to standardize numerical features so they have approximately zero mean and unit variance.

The scaler is fitted exclusively on the training dataset to prevent information leakage from the test dataset.

In [23]:
from sklearn.preprocessing import StandardScaler

#intialize scaler

scaler = StandardScaler()

In [24]:
#fit and transform training data

X_train_scaled = scaler.fit_transform(X_train)
print("Scaled training shape:", X_train_scaled.shape)

Scaled training shape: (455, 30)


In [25]:
#Transform test data

X_test_scaled = scaler.transform(X_test)
print("Scaled testing shape:", X_test_scaled.shape)

Scaled testing shape: (114, 30)


In [26]:
print("Mean:", X_train_scaled.mean(axis=0)[:5])
print("Standard deviation:", X_train_scaled.std(axis=0)[:5])

Mean: [-4.31742554e-15  2.24606658e-15 -7.38359313e-16  1.71779562e-16
  5.22695440e-15]
Standard deviation: [1. 1. 1. 1. 1.]


In [27]:
#Build Scikit-lean Pipeline

from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression


#Create ML pipeline

model_pipeline = Pipeline([
    ("scaler", StandardScaler()),
    ("classifier", LogisticRegression(
        max_iter=1000,
        random_state=42
    ))
])

print (model_pipeline)

Pipeline(steps=[('scaler', StandardScaler()),
                ('classifier',
                 LogisticRegression(max_iter=1000, random_state=42))])


Observation - Feature Scaling

StandardScaler was applier to standardize the numerical features using statistics calculated exclusively from the training dataset.

The transformed training features have approximately zero mean and unit variance.

A Sckit-lean Pipeline was also defined to combine feature scaling and Logistic Regression into a reproducible workflow. This ensures consistent preprocessing during training and prediction while preventing test-data leakage

Model Training - Logistic Regression 

A Logistic Regression classified is trained using the standardizewd numerical feautures.

The model is implemented through a Scikit-learn Pipeline that combines StandardScaler and LogisitcRegression.

Only the training dataset is used during fitting. the test dataset remains unseen until final evaluation


In [28]:
#Train the model using training data only

model_pipeline.fit(X_train,y_train)
print("Model training completed successfully!")

Model training completed successfully!


In [29]:
# Predict diagnosis for unseen test samples
y_pred = model_pipeline.predict(X_test)

print("First 10 predictions:")
print(y_pred[:10])

print("\nFirst 10 actual labels:")
print(y_test.iloc[:10].values)

First 10 predictions:
[0 1 0 1 0 1 1 0 0 0]

First 10 actual labels:
[0 1 0 1 0 1 1 0 0 0]


In [30]:
#Create a comparison DataFrame
results_df = pd.DataFrame({
    "Actual": y_test.to_numpy(),
    "Predicted":y_pred
})

#Map numeric labels to diagnosis names
diagnosis_map ={
    0:"Malignant",
    1:"Benign"
}

results_df["Actual"] = results_df["Actual"].map(diagnosis_map)
results_df["Predicted"] = results_df["Predicted"].map(diagnosis_map)

#Display first 10 results
results_df.head(10)


,Actual,Predicted
0,Malignant,Malignant
1,Benign,Benign
2,Malignant,Malignant
3,Benign,Benign
4,Malignant,Malignant
5,Benign,Benign
6,Benign,Benign
7,Malignant,Malignant
8,Malignant,Malignant
9,Malignant,Malignant


In [31]:
# Generate class probabilities
y_prob = model_pipeline.predict_proba(X_test)

print("Class order:", model_pipeline.classes_)
print("\nFist 5 probability predictions:")
print(y_prob[:5])

Class order: [0 1]

Fist 5 probability predictions:
[[9.99999941e-01 5.88824186e-08]
 [1.13351672e-05 9.99988665e-01]
 [9.93589175e-01 6.41082462e-03]
 [4.66491463e-01 5.33508537e-01]
 [9.99999999e-01 6.52500097e-10]]


In [36]:
# Access the trained classifier

classifier = model_pipeline.named_steps["classifier"]


# Extract feature coefficients

coefficients = pd.DataFrame({
    "Feature": X_train.columns,
    "Coefficient": classifier.coef_[0]
})

# Sort by absolute coefficient magnitude
coefficients["Absolute_Coefficient"] =(
    coefficients["Coefficient"].abs()
)

coefficients = coefficients.sort_values(
    by = "Absolute_Coefficient",
    ascending=False
)

coefficients.head(10)

,Feature,Coefficient,Absolute_Coefficient
21,worst texture,-1.255088,1.255088
10,radius error,-1.082965,1.082965
27,worst concave points,-0.953686,0.953686
23,worst area,-0.947756,0.947756
20,worst radius,-0.947616,0.947616
28,worst symmetry,-0.939181,0.939181
13,area error,-0.929104,0.929104
26,worst concavity,-0.823151,0.823151
22,worst perimeter,-0.763220,0.763220
24,worst smoothness,-0.746625,0.746625


Observation - Logistic Regression Model Training

A Logistic Regression classifier was successfully trained using the training dataset through a Scikit-learn Pipeline

The pipeline applies StandardScaler befoer classification, ensuring consistent preprocessing without introducing test-data leakage.

The trained model was used to generate class prediction and probability estimates fore a unseen test dataset.

Learned feature coefficients were also extracted to examine how individual standardized features contribute to the model's predictions

Formal performance evaluation will be performed in the next phase using classification metrics and a confustion matrix